In [1]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

  Using cached setuptools-78.1.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.m

## Config for BERT-WSD

In [ ]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: en-zh.en-filtered-wsd-processed.en.subword.train
        path_tgt: en-zh.zh-filtered-wsd.zh.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: en-zh.en-filtered-wsd-processed.en.subword.dev
        path_tgt: en-zh.zh-filtered-wsd.zh.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 10000
tgt_vocab_size: 10000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 512
src_seq_length: 512

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.fren

# Stop training if it does not improve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 2000

# To save space, limit checkpoints to last n
# keep_checkpoint: 3

seed: 3435

# Default: 100000 - Train the model to max n steps 
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 10000

# Default: 10000 - Run validation after n steps
valid_steps: 2000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 4000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"
weight_decay: 0.0001

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)

In [ ]:
# Find the number of CPUs/cores on the machine
!nproc --all

In [ ]:
# Build Vocabulary

# -config: path to your config.yaml file
# -n_sample: use -1 to build vocabulary on all the segment in the training dataset
# -num_threads: change it to match the number of CPUs to run it faster

!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 20

In [ ]:
# Check if the GPU is active
!nvidia-smi -L

In [ ]:
# Check if the GPU is visable to PyTorch
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

gpu_memory = torch.cuda.mem_get_info(0)
print("Free GPU memory:", gpu_memory[0]/1024**2, "out of:", gpu_memory[1]/1024**2)

In [ ]:
# Train the NMT model
!onmt_train -config config.yaml

## Perform Training on SoC Computer Cluster

1. **SSH to your SoC Computer Cluster and copy the train dev test files and config.yaml over**  

2. **Run `salloc` to acquire a GPU host:**  
   ```bash
   salloc -G nv -p gpu-long

3. **Find number of cores available:**  
   ```bash
   srun nproc --all

4. **Build vocabulary according to number of cores available:**  
   ```bash
   srun onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 20

5. **Train it on `slurm` by setting time limit to one day:** 
    ```bash
    srun -t 1440 onmt_train -config config.yaml

## Translate

In [ ]:
# Translate the "subworded" source file of the test dataset
# Change the model name, if needed.
!onmt_translate -model models/model.fren_step_10000.pt -src en-zh.en-filtered-wsd-processed.en.subword.test -output zh.translated -gpu 0 -min_length 1

In [2]:
# Check the first 5 lines of the translation file
!head -n 30 zh.translated

▁在 克 萨 斯 洛 伐 克 , 东 德国 , ▁ 欧 沙 尼亚 , 拉 维 亚 , 里 南 , ▁ 利 萨 维 亚 , 马 里 兰 , ▁ 奥 地 利 , 奥 地 利 , ▁ 奥 里 兰 , 塞 维 亚 , 我 还可以 继续 , ▁ 继续 努力 。
▁我不知道 他们 要 用 这些东西 做什么
▁它 需要 某种 交通 系统 来 规划 , ▁因为它 是一个 没有 交通 系统 的 系统 。
▁ 大 公司 会 这样做 , 但是 ▁ 大 公司 也 做 得很好 。
▁所以 , 我 只是 用 这种 排 骨 和 甘 地 的 薄 板 ▁ 插 到 最后 , 在 4 0 0 0 英里 的 地 上 , ▁当我 靠近 我的 神圣 门 时 , ▁我发现 蛋白质 。
▁我 注意到 他们会 移动 这个 微 缩 设备 , ▁ 用来 改变 家 的 温度 , ▁ 一 到 两 英寸 。
▁但是 这是一个 谜 .
▁ 平均 年龄 是 3 2 岁 , ▁ 正常 年龄 的人 的 平均 年龄 是 3 2 岁 。 ▁但是 , 当 你们 从 3 0 岁 到 6 0 岁 —— ▁ 6 0 岁 的人 —— ▁ 6 0 岁 以上 的人 在 使用 代 数 自行车 , ▁这 当然 不是 很 合 常 用 的 —— ▁我们 每 月 都会 增加 4 0 % ,
▁所以 它 继续 思考 着 大声 的 表达 出来 。
▁它 的 好处 在于 ▁它 允许 手机 开始 看到 ▁然后 了解 人类 大脑的 运作 方式
▁ 建筑师 的 职业 是 会 员 , ▁ 居民 实际上 把它 从 地 上 提高 。
▁然后 学生 走进 我们 的声音 办公室 , ▁他们 用 自己的 模型 做出 自己的 模型 。
▁有时候 我 有点 太 老 太 太 太 太 太 太 太 太 太 太 晚 了 , ▁ 留下 了 无 神 论 者 的 图画 。
▁如果你 填 上 小 额 信 贷 的 一杯 , ▁你就 产生了 一次 革命 。
▁然后 , 在 飓 风 后 , 她 问 他 , ▁“ 你 是谁 ?”
▁现在 , 我要 非常 小心 地 把 轮 子 组合 起来 。
▁它 是个 有 功能 的 系统 ▁尽管 有 计划 等等 , 它 也 进化 了 。
▁ 现代 人 走出 非洲 , ▁ 走出 非洲 , 来到 中东 。
▁这些 魔 法 帮助我们 解决问题 , ▁ 帮助我们 变得更 有 创造力 。

In [3]:
# If needed install/update sentencepiece
!pip3 install --upgrade -q sentencepiece

# Desubword the translation file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model zh.translated 

# Desubword test file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model en-zh.zh-filtered-wsd.zh.subword.test


Done desubwording! Output: zh.translated.desubword
Done desubwording! Output: en-zh.zh-filtered-wsd.zh.subword.test.desubword


In [1]:
# Desubword the target file (reference) of the test dataset
# Note: You might as well have split files *before* subwording during dataset preperation, 
# but sometimes datasets have tokeniztion issues, so this way you are sure the file is really untokenized.
!python3 ./MT-Preparation/subwording/3-desubword.py ./source.model en-zh.en-filtered-wsd.en.subword.test

Done desubwording! Output: en-zh.en-filtered-wsd.en.subword.test.desubword


In [4]:
# Check the first 5 lines of the desubworded translation file
!echo "---zh.translated.desubword---" # w/o weight decay
!head -n 5 zh.translated.desubword

# Check the first 5 lines of the desubworded reference
!echo "---en-zh.zh-filtered-wsd.zh.subword.test.desubword---"
!head -n 5 en-zh.zh-filtered-wsd.zh.subword.test.desubword

---zh.translated.desubword---
在克萨斯洛伐克,东德国, 欧沙尼亚,拉维亚,里南, 利萨维亚,马里兰, 奥地利,奥地利, 奥里兰,塞维亚,我还可以继续, 继续努力。
我不知道他们要用这些东西做什么
它需要某种交通系统来规划, 因为它是一个没有交通系统的系统。
大公司会这样做,但是 大公司也做得很好。
所以,我只是用这种排骨和甘地的薄板 插到最后,在4000英里的地上, 当我靠近我的神圣门时, 我发现蛋白质。
---en-zh.zh-filtered-wsd.zh.subword.test.desubword---
在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
我不知道他们将怎么处理那些东西。
这需要有探求精神 因为这整个系统不是以雕塑形式做成的
优秀的教师的确要这样做 但同时他们还会指导学生的学习 激发学生的兴趣,挑起学生的热情,赢得学生的关注
我按部就班地执行艰巨的任务 终于在第4000次筛选的时候 在我快发疯的时候 找到了符合标准的蛋白质


## Evaluation

In [13]:
# Download the BLEU script
!wget https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py

--2025-04-06 15:31:19--  https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 957 [text/plain]
Saving to: ‘compute-bleu.py’

compute-bleu.py     100%[===================>]     957  --.-KB/s    in 0s      

2025-04-06 15:31:19 (13.9 MB/s) - ‘compute-bleu.py’ saved [957/957]



In [14]:
# Install sacrebleu
!pip3 install sacrebleu

In [5]:
# Evaluate the translation (without subwording)
!python3 compute-bleu.py en-zh.zh-filtered-wsd.zh.subword.test.desubword zh.translated.desubword


Reference 1st sentence: 在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
MTed 1st sentence: 在克萨斯洛伐克,东德国, 欧沙尼亚,拉维亚,里南, 利萨维亚,马里兰, 奥地利,奥地利, 奥里兰,塞维亚,我还可以继续, 继续努力。
BLEU:  2.3900016239366977
